# Converting the ASHI Consensus Benchmark to Extended HAML

The [ASHI Consensus Benchmark](https://hlabassist.app) is a curated set of 55 HLA antibody cases:
40 antibody identification (AbID) cases and 15 virtual crossmatch (VXM) cases drawn from ASHI
proficiency testing materials. Every case has at least one SAB run in One Lambda HLA Fusion format.
A subset of cases also has a paired Werfen MATCH IT! run.

Converting the benchmark to a single multi-patient Extended HAML file serves two purposes:

1. **External evaluation** — any algorithm can run against the same standardized input without
   reverse-engineering our CSV preprocessing.
2. **Format demonstration** — the benchmark file is a real-world example of Extended HAML at scale
   (55 cases, 61 patient entries, ~6.7 MB).

This notebook walks through the conversion logic using the demo sample CSVs as stand-ins for the
benchmark data. All design decisions shown here apply directly to the production benchmark file
`hlabassist_benchmark.haml.xml` (available at [HLAbAssist.app](https://hlabassist.app) after login).

In [ ]:
import sys
from pathlib import Path
import lxml.etree as etree

# Find repo root (works from notebooks/ or repo root)
REPO_ROOT = Path.cwd()
if (REPO_ROOT / 'notebooks').exists():
    pass  # already at repo root
elif REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.csv_to_haml import read_fusion_csv, sub

HAML_NS = 'urn:HAML.Namespace'
NSMAP = {None: HAML_NS}

print(f'Repo root: {REPO_ROOT}')

---
## Part 1: Source data — one CSV per case, one row per bead

Each benchmark case arrives as a normalized CSV with these columns:
`Specificity`, `NormalValue` (background-corrected MFI), `RawMFI`,
`NegativeControl`, `PositiveControl`, `vendor`, `Description`.

The demo CSV uses the One Lambda HLA Fusion column names (`Specificity`, `Raw Value`, `BCM`).
The production benchmark converter normalizes both One Lambda and Werfen column names into this
common schema before building HAML.

In [ ]:
import pandas as pd

csv_path = REPO_ROOT / 'data' / 'sample_sab_class1.csv'
df = pd.read_csv(csv_path)
print(f'Loaded {len(df)} rows from {csv_path.name}')
print(f'Columns: {list(df.columns)}')
df.head(6)

---
## Part 2: Benchmark-specific design decisions

Converting 55 cases into a single multi-patient HAML file requires three decisions that don't arise
for single-patient conversion.

### Decision 1: Multi-vendor cases — separate `<patient>` entries with vendor suffixes

Some benchmark cases have both a One Lambda and a Werfen run. The two platforms have different bead
catalogs, different NC bead identifiers, and different MFI normalization conventions. Merging them
into a single `<assay>` would require custom logic to pair NC values with the correct set of beads.

The solution: each (case × vendor) combination becomes its own `<patient>` entry, distinguished by
a suffix on the `<patient-id>`:

| Suffix | Platform |
|---|---|
| `_ol` or `_onelambda` | One Lambda (LABScreen) |
| `_im` or `_werfen` | Werfen (LIFECODES) |

The benchmark pipeline strips the suffix to group entries by case before evaluation.

### Decision 2: Historic timepoints — separate `<patient>` entries with `_historic` suffix

Five VXM cases have a second SAB run from an earlier timepoint (pre-sensitization). Storing the
historic run as a second `<patient>` entry (e.g., `VXM-001_historic`) keeps single-timepoint parsing
simple. The benchmark harness groups by base ID and passes both timepoints to the algorithm.

### Decision 3: Multi-assay per working-sample — one assay per (vendor × NC-group)

Within a One Lambda run, Class I and Class II beads have different NC beads. One `<assay>` element
is created per NC-group (usually per class), not per panel. This preserves the per-class NC value
so the algorithm can apply adaptive per-class thresholds.

---
## Part 3: Building a multi-patient HAML

Here we simulate converting two benchmark cases into a single HAML file. Case A uses only One Lambda.
Case B has both One Lambda and Werfen runs, demonstrating the vendor-suffix pattern.

In [ ]:
from datetime import datetime

def build_assay(parent, beads_df, manufacturer='One Lambda',
                lot='DEMO-001', catalog='LS1A04', hla_class='I'):
    """Add one <assay> element to parent with beads from beads_df."""
    assay = etree.SubElement(parent, f'{{{HAML_NS}}}assay')

    # --- kit metadata ---
    kit = sub(assay, 'assay-kit')
    sub(kit, 'kit-manufacturer', manufacturer)
    sub(kit, 'lot-number', lot)
    sub(kit, 'catalog-number', catalog)
    sub(kit, 'assay-type', f'Class {hla_class} Single Antigen Bead')

    for _, row in beads_df.iterrows():
        obs = etree.SubElement(assay, f'{{{HAML_NS}}}target-bead-observation')

        # bead-info
        info = sub(obs, 'bead-info')
        bead_id = str(row.get('Bead ID', ''))
        sub(info, 'bead-id', bead_id)

        spec = str(row.get('Specificity', '')).strip()
        if bead_id == '1':
            sub(info, 'bead-type', 'negative-control')
        elif bead_id == '2':
            sub(info, 'bead-type', 'positive-control')
        elif spec:
            sub(info, 'bead-type', 'target')
            sub(info, 'HLA-target-type', spec)
        else:
            continue  # skip unknown beads

        # bead-raw-data
        raw_data = sub(obs, 'bead-raw-data')
        # BCM (background-corrected MFI) → <raw-MFI> following HAML convention
        bcm = row.get('BCM', row.get('Raw Value', 0))
        sub(raw_data, 'raw-MFI', bcm)
        if row.get('Bead Count'):
            sub(raw_data, 'bead-count', int(row['Bead Count']))

        # <extended-bead-data> for raw instrument count and ranking
        ext_attrs = {}
        if 'Raw Value' in row and not pd.isna(row['Raw Value']):
            ext_attrs['raw-MFI-unprocessed'] = str(row['Raw Value'])
        if 'Ranking' in row and str(row.get('Ranking','')).strip():
            ext_attrs['ranking'] = str(row['Ranking'])
        if ext_attrs:
            etree.SubElement(obs, 'extended-bead-data', attrib=ext_attrs)

    return assay


def add_patient(haml_root, patient_id, beads_df, manufacturer='One Lambda',
                lot='DEMO-001', catalog='LS1A04', hla_class='I'):
    """Add one <patient> element to the <haml> root."""
    patient = etree.SubElement(haml_root, f'{{{HAML_NS}}}patient')
    sub(patient, 'patient-id', patient_id)
    sample = sub(patient, 'sample')
    ws = sub(sample, 'working-sample')
    build_assay(ws, beads_df, manufacturer=manufacturer, lot=lot, catalog=catalog, hla_class=hla_class)
    return patient


# --- Load demo data ---
c1 = pd.read_csv(REPO_ROOT / 'data' / 'sample_sab_class1.csv')
c2 = pd.read_csv(REPO_ROOT / 'data' / 'sample_sab_class2.csv')

# Simulate a Werfen run for Case B by using Class II data with a different vendor name
c2_werfen = c2.copy()
c2_werfen['vendor'] = 'Werfen'

# --- Build multi-patient HAML ---
ext_root = etree.Element('extended-haml', attrib={
    'version': '1.0',
    'created': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
})

haml_root = etree.SubElement(ext_root, f'{{{HAML_NS}}}haml', nsmap=NSMAP, attrib={'version': '0.5.3'})

# Case A: One Lambda only (AC-001_ol)
add_patient(haml_root, 'AC-001_ol', c1, manufacturer='One Lambda', catalog='LS1A04', hla_class='I')

# Case B: One Lambda (Class I) + Werfen (Class II) — two separate patient entries
add_patient(haml_root, 'AC-002_ol', c1, manufacturer='One Lambda', catalog='LS1A04', hla_class='I')
add_patient(haml_root, 'AC-002_im', c2, manufacturer='Werfen', catalog='LifeScreen', hla_class='II')

patient_ids = [p.findtext(f'{{{HAML_NS}}}patient-id') for p in haml_root.findall(f'{{{HAML_NS}}}patient')]
print(f'Built {len(patient_ids)} patient entries: {patient_ids}')

### Adding recipient and donor HLA typing

The VXM cases in the benchmark include donor typing for virtual crossmatch prediction.
For AbID cases, only recipient typing is present (no donor).
Both are stored as sibling elements to `<haml>` — outside the patient data.

In [ ]:
# Add recipient HLA typing (required for self-antigen exclusion)
recip = etree.SubElement(ext_root, 'recipient-profile', attrib={'format': 'molecular'})
etree.SubElement(recip, 'alleles').text = 'A*01:01, A*24:02, B*07:02, B*44:02, DRB1*04:01, DRB1*15:01'

# Add donor typing (VXM cases only — omit for AbID cases)
donor = etree.SubElement(ext_root, 'donor-profile', attrib={'format': 'molecular'})
etree.SubElement(donor, 'alleles').text = 'A*02:01, A*24:02, B*44:02, B*57:01, DRB1*04:01, DRB1*07:01'

print('recipient-profile and donor-profile added.')

# Serialize and show structure
xml_bytes = etree.tostring(ext_root, pretty_print=True, xml_declaration=True, encoding='UTF-8')
lines = xml_bytes.decode().split('\n')
# Print structure: first 50 lines
for line in lines[:50]:
    print(line)

---
## Part 4: Validate and write

The inner `<haml>` element is valid HAML 0.5.3. We validate it against the XSD schema,
then write the full Extended HAML file.

In [ ]:
from scripts.csv_to_haml import validate_haml

# Validate just the inner <haml> element
is_valid, errors = validate_haml(haml_root)
if is_valid:
    print('Inner <haml>: schema validation PASSED')
else:
    print(f'Validation errors ({len(errors)}):')
    for e in errors:
        print(f'  {e}')

# Write full Extended HAML
out_path = REPO_ROOT / 'output' / 'benchmark_demo.haml.xml'
out_path.parent.mkdir(exist_ok=True)
out_path.write_bytes(xml_bytes)
print(f'\nWrote {out_path} ({out_path.stat().st_size:,} bytes)')

# Summary
all_patients = haml_root.findall(f'{{{HAML_NS}}}patient')
all_beads = haml_root.findall(f'.//{{{HAML_NS}}}target-bead-observation')
print(f'{len(all_patients)} patient entries, {len(all_beads)} bead observations')

---
## Part 5: Reading the benchmark file back

The benchmark pipeline reads the multi-patient HAML, strips vendor suffixes to group entries by
case ID, and routes each group to the appropriate evaluation path.

In [ ]:
import re

VENDOR_SUFFIXES = ['_ol', '_onelambda', '_im', '_werfen', '_historic']

def strip_suffix(patient_id):
    """Return (base_id, suffix) for a benchmark patient ID."""
    for sfx in VENDOR_SUFFIXES:
        if patient_id.endswith(sfx):
            return patient_id[:-len(sfx)], sfx
    return patient_id, ''

def parse_benchmark_haml(haml_path):
    """Read a benchmark HAML file and group patient entries by case ID."""
    tree = etree.parse(str(haml_path))
    root = tree.getroot()

    # Support both plain HAML and Extended HAML roots
    ns = HAML_NS
    haml_el = root.find(f'{{{ns}}}haml') if root.tag == 'extended-haml' else root

    cases = {}
    for patient in haml_el.findall(f'{{{ns}}}patient'):
        pid = patient.findtext(f'{{{ns}}}patient-id', default='')
        base_id, suffix = strip_suffix(pid)
        cases.setdefault(base_id, []).append({'patient_id': pid, 'suffix': suffix})

    return cases

cases = parse_benchmark_haml(out_path)
print(f'Found {len(cases)} cases in benchmark file:\n')
for case_id, entries in sorted(cases.items()):
    entry_desc = ', '.join(e['suffix'] or '(no suffix)' for e in entries)
    print(f'  {case_id}: {entry_desc}')

---
## Summary

The benchmark conversion produces a file where:

- Each case is one or more `<patient>` entries grouped by base ID.
- Multi-vendor cases carry `_ol` / `_im` suffixes — the parser groups them at runtime.
- Historic timepoints carry `_historic` suffixes — the harness passes both to the algorithm.
- Class I and Class II beads from the same run are stored in separate `<assay>` elements (one per NC-group), preserving the per-class NC value for adaptive threshold computation.
- The `<extended-bead-data>` element preserves the raw Luminex instrument count alongside the BCM value that the algorithm actually uses.
- Recipient and donor HLA typing are stored as sibling elements to `<haml>`, not nested inside `<patient>`.

The production benchmark file (`hlabassist_benchmark.haml.xml`) follows the same structure at
scale: 55 cases, 61 patient entries, covering all ASHI AbID and VXM proficiency testing scenarios.

See [`docs/extended_haml_schema.md`](../docs/extended_haml_schema.md) for the full schema reference
and [`docs/hlabassist_workflow.md`](../docs/hlabassist_workflow.md) for the end-to-end data flow.